In [2]:
import pandas as pd



In [3]:
df = pd.read_csv("../data/raw_data/socioeconomic_factors/time_series_socio_factors/Ausländeranteil_1978–2025_ZH_per_mun.csv")

df.head()

,BFS_NR,GEBIET_NAME,THEMA_NAME,SET_NAME,SUBSET_NAME,INDIKATOR_ID,INDIKATOR_NAME,INDIKATOR_JAHR,INDIKATOR_VALUE,EINHEIT_KURZ,EINHEIT_LANG,Unnamed: 11
0,1,Aeugst a.A.,Bevölkerung und Soziales,Bevölkerungsbestand und Haushalte,Bevölkerungsbestand,205,Ausländeranteil [%],1978,3.1,%,Prozent,NaN
1,1,Aeugst a.A.,Bevölkerung und Soziales,Bevölkerungsbestand und Haushalte,Bevölkerungsbestand,205,Ausländeranteil [%],1979,3.5,%,Prozent,NaN
2,1,Aeugst a.A.,Bevölkerung und Soziales,Bevölkerungsbestand und Haushalte,Bevölkerungsbestand,205,Ausländeranteil [%],1980,3.8,%,Prozent,NaN
3,1,Aeugst a.A.,Bevölkerung und Soziales,Bevölkerungsbestand und Haushalte,Bevölkerungsbestand,205,Ausländeranteil [%],1981,3.3,%,Prozent,NaN
4,1,Aeugst a.A.,Bevölkerung und Soziales,Bevölkerungsbestand und Haushalte,Bevölkerungsbestand,205,Ausländeranteil [%],1982,3.6,%,Prozent,NaN


In [4]:
print(df.shape)
print(df.columns)
print(df.isna().sum())

(9433, 12)
Index(['BFS_NR', 'GEBIET_NAME', 'THEMA_NAME', 'SET_NAME', 'SUBSET_NAME',
       'INDIKATOR_ID', 'INDIKATOR_NAME', 'INDIKATOR_JAHR', 'INDIKATOR_VALUE',
       'EINHEIT_KURZ', 'EINHEIT_LANG', 'Unnamed: 11'],
      dtype='str')
BFS_NR                0
GEBIET_NAME           0
THEMA_NAME            0
SET_NAME              0
SUBSET_NAME           0
INDIKATOR_ID          0
INDIKATOR_NAME        0
INDIKATOR_JAHR        0
INDIKATOR_VALUE       0
EINHEIT_KURZ          0
EINHEIT_LANG          0
Unnamed: 11        9433
dtype: int64


In [5]:
df = df.drop(columns=["Unnamed: 11"])

print(df.shape)

(9433, 11)


In [6]:
print(df.columns)


Index(['BFS_NR', 'GEBIET_NAME', 'THEMA_NAME', 'SET_NAME', 'SUBSET_NAME',
       'INDIKATOR_ID', 'INDIKATOR_NAME', 'INDIKATOR_JAHR', 'INDIKATOR_VALUE',
       'EINHEIT_KURZ', 'EINHEIT_LANG'],
      dtype='str')


In [7]:
print(df[df["BFS_NR"] == 0]["GEBIET_NAME"].unique()) #names of row where bfs == 0, which are non-municipalities

<ArrowStringArray>
[         'Bezirk Affoltern',        'Bezirk Andelfingen',
             'Bezirk Bülach',          'Bezirk Dielsdorf',
           'Bezirk Dietikon',             'Bezirk Hinwil',
             'Bezirk Horgen',             'Bezirk Meilen',
          'Bezirk Pfäffikon',              'Bezirk Uster',
         'Bezirk Winterthur',             'Bezirk Zürich',
            'Region Furttal',            'Region Glattal',
        'Region Knonaueramt',          'Region Limmattal',
           'Region Oberland',       'Region Pfannenstiel',
          'Region Unterland',           'Region Weinland',
 'Region Winterthur u. Umg.',         'Region Zimmerberg',
             'Region Zürich',    'Zürich - ganzer Kanton']
Length: 24, dtype: str


In [8]:
df = df[df["BFS_NR"] != 0] #drop rows where bfs == 0
print(df.shape)

(8281, 11)


In [9]:
print(df[df["GEBIET_NAME"].str.contains("bis")]["GEBIET_NAME"].unique()) #muns have merged since 2014, old muns are still in table with entry containing "bis" 
#I now look at these muns which have merged

<ArrowStringArray>
[          'Adlikon (bis 2022)',       'Andelfingen (bis 2022)',
          'Humlikon (bis 2022)',     'Oberstammheim (bis 2018)',
    'Unterstammheim (bis 2018)',       'Waltalingen (bis 2018)',
            'Hirzel (bis 2017)',            'Horgen (bis 2017)',
            'Hütten (bis 2018)',       'Schönenberg (bis 2018)',
         'Wädenswil (bis 2018)',             'Bauma (bis 2014)',
 'Illnau-Effretikon (bis 2015)',            'Kyburg (bis 2015)',
       'Sternenberg (bis 2014)',              'Elgg (bis 2017)',
        'Hofstetten (bis 2017)']
Length: 17, dtype: str


In [10]:
register = pd.read_csv("../data/raw_data/geographic_reference/bfs_municipalities/BFS_Gemeindeverzeichnis_2026-01-01.csv")
register.head()
#register lists all muns which still exist in 2026

,HistoricalCode,BfsCode,ValidFrom,ValidTo,Level,Parent,Name,ShortName,Inscription,Radiation,Rec_Type_fr,Rec_Type_de
0,1,1,12.09.1848,NaN,1,NaN,Zürich,ZH,NaN,NaN,NaN,NaN
1,10053,101,12.09.1848,NaN,2,1.0,Bezirk Affoltern,Affoltern,100.0,NaN,NaN,NaN
2,10575,13,12.09.1848,NaN,3,10053.0,Stallikon,Stallikon,NaN,NaN,NaN,NaN
3,11742,2,12.09.1848,NaN,3,10053.0,Affoltern am Albis,Affoltern am Albis,NaN,NaN,NaN,NaN
4,11801,3,12.09.1848,NaN,3,10053.0,Bonstetten,Bonstetten,NaN,NaN,NaN,NaN


In [12]:
zh_gemeinden = register[(register["Level"] == 3) & (register["BfsCode"] <= 298)]

print(len(zh_gemeinden))
#filter for muns via Level which and for lying in Zurich via BfsCode

160


In [13]:
df = df[df["BFS_NR"].isin(zh_gemeinden["BfsCode"])]
print(df.shape)
print(df["BFS_NR"].nunique())
#just keep muns from df which are contained in register, checked via code

(7590, 11)
160


In [14]:
print(df[df["GEBIET_NAME"].str.contains("bis")]["GEBIET_NAME"].unique())
#check  whether mun containing bis in name is still contained

<ArrowStringArray>
[]
Length: 0, dtype: str


In [15]:
print(df["INDIKATOR_NAME"].unique())
print(df["EINHEIT_KURZ"].unique())
#units of werte ausländeranteil

<ArrowStringArray>
['Ausländeranteil [%]']
Length: 1, dtype: str
<ArrowStringArray>
['%']
Length: 1, dtype: str


In [16]:
df = df[["BFS_NR", "INDIKATOR_JAHR", "INDIKATOR_VALUE"]]

df.head()
#keep relevant columns

,BFS_NR,INDIKATOR_JAHR,INDIKATOR_VALUE
0,1,1978,3.1
1,1,1979,3.5
2,1,1980,3.8
3,1,1981,3.3
4,1,1982,3.6


In [17]:
df = df.rename(columns={
    "BFS_NR": "GDENR",
    "INDIKATOR_JAHR": "year",
    "INDIKATOR_VALUE": "foreigner_rate"
})

df.head()
#rename columns such that match with popular participation file

,GDENR,year,foreigner_rate
0,1,1978,3.1
1,1,1979,3.5
2,1,1980,3.8
3,1,1981,3.3
4,1,1982,3.6


In [18]:
print(df.dtypes)
#check types

GDENR               int64
year                int64
foreigner_rate    float64
dtype: object


In [20]:
print(df.duplicated(subset=["GDENR", "year"]).sum())    
#how many duplicate rows determined by entries GDNER AND year

0


In [21]:
print(df["foreigner_rate"].describe())
#do entries make sense?

count    7590.000000
mean       15.025903
std         8.169794
min         0.000000
25%         8.800000
50%        14.000000
75%        20.175000
max        48.800000
Name: foreigner_rate, dtype: float64


In [22]:
print(df["year"].min(), df["year"].max())
#do entries make sense

1978 2025


In [23]:
print(df.groupby("year")["GDENR"].count())
#amount of rows per bucket

year
1978    155
1979    155
1980    155
1981    155
1982    155
1983    155
1984    155
1985    155
1986    155
1987    155
1988    155
1989    155
1990    155
1991    155
1992    155
1993    155
1994    155
1995    155
1996    160
1997    160
1998    160
1999    160
2000    160
2001    160
2002    160
2003    160
2004    160
2005    160
2006    160
2007    160
2008    160
2009    160
2010    160
2011    160
2012    160
2013    160
2014    160
2015    160
2016    160
2017    160
2018    160
2019    160
2020    160
2021    160
2022    160
2023    160
2024    160
2025    160
Name: GDENR, dtype: int64


In [24]:
df.head()

,GDENR,year,foreigner_rate
0,1,1978,3.1
1,1,1979,3.5
2,1,1980,3.8
3,1,1981,3.3
4,1,1982,3.6


In [25]:

df.to_csv("../data/clean_data/zh_auslaenderanteil.csv", index=False)